# Practice Lab: Human-in-the-Loop Patterns (Lesson 11.10)

Module 11 . 8 exercises . interrupt() + tiers + confidence + escalation + needsApproval + Temporal + autonomy + audit


# Lesson 11.10: Human-in-the-Loop Patterns

8 exercises: 2E + 3M + 3C

Exercises 1-5 run **locally** (pure Python). Ex 6-8 are design.


In [ ]:
import json
import numpy as np
from datetime import datetime, timedelta
from collections import Counter
np.random.seed(42)


---
## Exercise 1 (Easy): LangGraph interrupt()



In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
class SimInterrupt(Exception):
    def __init__(self,p): self.payload=p

class Node:
    def __init__(self): self.exec_count=0; self.resume=None
    def interrupt(self,payload):
        if self.resume is not None: v=self.resume; self.resume=None; return v
        raise SimInterrupt(payload)
    def approval(self,state):
        self.exec_count+=1; action=state["action"]
        print(f"    [exec #{self.exec_count}] Checking: {action}")
        if action in ["delete","purchase","send_email","refund"]:
            approved=self.interrupt({"action":action,"risk":"HIGH"})
            if approved: print(f"    [exec #{self.exec_count}] APPROVED"); return {"status":"executed"}
            else: print(f"    [exec #{self.exec_count}] REJECTED"); return {"status":"rejected"}
        print(f"    [exec #{self.exec_count}] Safe -> auto"); return {"status":"auto_executed"}

print("LangGraph interrupt():")
n=Node(); print("  Safe:"); n.approval({"action":"search"})
n2=Node(); print("  Dangerous (approve):")
try: n2.approval({"action":"purchase"})
except SimInterrupt as e:
    print(f"    INTERRUPTED: {e.payload}")
    n2.resume=True; n2.approval({"action":"purchase"})
n3=Node(); print("  Dangerous (reject):")
try: n3.approval({"action":"delete"})
except SimInterrupt as e:
    n3.resume=False; n3.approval({"action":"delete"})
print("\n  Key: node re-runs on resume. Code before interrupt() must be idempotent")


</details>


---
## Exercise 2 (Easy): Three-Tier Action Classifier



In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
class Classifier:
    TIERS={"safe":["search","read","calculate","summarize","translate","list"],
           "risky":["edit","submit_form","send_notification","update_record"],
           "dangerous":["delete","purchase","process_refund","send_mass_email"]}
    def classify(self,a):
        for t,acts in self.TIERS.items():
            if a in acts: return t
        return "risky"
    def execute(self,a,approvers=None):
        t=self.classify(a)
        if t=="safe": return f"auto_executed"
        if t=="risky": return f"approved by {approvers[0]}" if approvers and len(approvers)>=1 else "pending_single"
        return f"approved by {approvers}" if approvers and len(approvers)>=2 else f"pending_multi(need 2)"

c=Classifier()
print("Three-Tier Action Classifier:")
counts={"safe":0,"risky":0,"dangerous":0}
for a in ["search","read","calculate","edit","submit_form","delete","purchase","send_notification","summarize","process_refund"]:
    t=c.classify(a); counts[t]+=1; print(f"  {a:<22} -> {t}")
total=sum(counts.values())
print(f"\n  Distribution: {', '.join(f'{k}:{v}/{total}({v/total*100:.0f}%)' for k,v in counts.items())}")
print(f"  Multi-person: delete(0)={c.execute('delete')} | delete(2)={c.execute('delete',['alice','bob'])}")


</details>


---
## Exercise 3 (Medium): Confidence Router



In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import numpy as np; np.random.seed(42)
from collections import Counter

class ConfRouter:
    def __init__(self,th=0.8): self.th=th; self.stats={"auto":0,"esc":0}
    def ensemble(self,q):
        if any(w in q.lower() for w in ["price","cost","hours"]): answers=["14999"]*5
        elif any(w in q.lower() for w in ["visa","replace","legal"]): answers=["Maybe","No","Yes","Depends","Uncertain"]
        else: n=np.random.randint(3,6); answers=["A"]*n+["B"]*(5-n)
        ct=Counter(answers); agree=ct.most_common(1)[0][1]/5
        return agree
    def route(self,q):
        a=self.ensemble(q)
        if a>=self.th: self.stats["auto"]+=1; return "AUTO",a
        self.stats["esc"]+=1; return "HUMAN",a

r=ConfRouter(0.8)
print("Confidence Router (Ensemble):")
for q in ["GenAI course price?","How many hours?","Can I use for US visa?","Will GenAI replace my job?","Refund policy?","This or Coursera?"]:
    route,agree=r.route(q); print(f"  {q[:35]:<35} {agree:.0%} -> {route}")

print(f"\n  Compound uncertainty: " + " | ".join(f"{s:.0%}->{s**3:.0%}" for s in [0.95,0.90,0.85,0.80]))
total=r.stats["auto"]+r.stats["esc"]; print(f"  Escalation: {r.stats['esc']}/{total} ({r.stats['esc']/total:.0%}). Target: 10-15%")


</details>


---
## Exercise 4 (Medium): SLA Escalation Chain



In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
class EscChain:
    TIERS=[{"name":"AI","sla":0},{"name":"L1","sla":2},{"name":"L2","sla":4},{"name":"Emergency","sla":0.5}]
    def __init__(self): self.stats={t["name"]:0 for t in self.TIERS}
    def route(self,task,conf,risk):
        if conf>=0.9 and risk=="low": tier=0
        elif conf>=0.7 or risk=="medium": tier=1
        elif risk=="high": tier=2
        else: tier=3
        self.stats[self.TIERS[tier]["name"]]+=1
        return self.TIERS[tier]["name"],self.TIERS[tier]["sla"]
    def timeout(self,tier_name,hours):
        idx=next(i for i,t in enumerate(self.TIERS) if t["name"]==tier_name)
        if hours>self.TIERS[idx]["sla"] and idx<len(self.TIERS)-1:
            return f"ESCALATED {tier_name} -> {self.TIERS[idx+1]['name']}"
        return f"within SLA"

chain=EscChain()
print("SLA Escalation Chain:")
for task,conf,risk in [("Search price",0.95,"low"),("Process refund",0.6,"high"),("Update email",0.85,"medium"),
    ("Delete account",0.3,"high"),("Summarize",0.92,"low"),("Mass email",0.5,"high")]:
    tier,sla=chain.route(task,conf,risk); print(f"  {task:<18} [{conf:.2f},{risk:>6}] -> {tier} (SLA:{sla}h)")

print(f"\n  Timeout: L2 at 1h={chain.timeout('L2',1)} | 5h={chain.timeout('L2',5)}")
total=sum(chain.stats.values())
esc_rate=1-chain.stats["AI"]/total
print(f"  Stats: {chain.stats} | Escalation: {esc_rate:.0%} (target: 10-15%)")


</details>


---
## Exercise 5 (Medium): OpenAI needsApproval



In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
class SimApproval:
    def __init__(self): self.interrupts=[]; self.sticky={}; self.results=[]
    def call(self,tool,args,needs_approval=False):
        if tool in self.sticky: return f"{self.sticky[tool]} (sticky)"
        if needs_approval: self.interrupts.append(tool); return "interrupted"
        return "auto_executed"
    def approve(self,tool,always=False):
        self.interrupts=[i for i in self.interrupts if i!=tool]
        if always: self.sticky[tool]="approved"
        return "approved"
    def reject(self,tool): self.interrupts=[i for i in self.interrupts if i!=tool]; return "rejected"

a=SimApproval()
print("OpenAI needsApproval:")
for tool,args,na in [("search",{},False),("send_email",{"to":"x"},True),("purchase",{"item":"course"},True),("calc",{},False)]:
    print(f"  {tool:<14} needs={na} -> {a.call(tool,args,na)}")
print(f"  Pending: {a.interrupts}")

a.approve("send_email",always=True); a.reject("purchase")
print(f"  send_email: {a.approve('send_email',True)} | purchase: rejected")
print(f"  Sticky test: {a.call('send_email',{},True)}")
print(f"\n  OpenAI: needsApproval + sticky + interruptions")
print(f"  Anthropic: tool_use interception + can_use_tool callback + 5 modes")


</details>


---
## Exercise 6 (Challenge): Temporal Durable HITL
Design/architecture.


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
print("Temporal Durable HITL:")
print("  Signal: workflow.wait_condition(lambda: self.approved, timeout=1h)")
print("  Durable timer survives crashes/deploys")
print("  Crash recovery: approval preserved via event replay")
print("  Multi-approver: wait for sum(approvals.values()) >= 2")
print("  Gold standard: Temporal > LangGraph for durable HITL")


</details>


---
## Exercise 7 (Challenge): Progressive Autonomy
Design/architecture.


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
class PA:
    def __init__(self): self.stage="audit"; self.trust=0; self.m={"total":0,"correct":0,"overrides":0,"critical":0}
    def process(self,ok=True,ov=False,cr=False):
        self.m["total"]+=1
        if ok: self.m["correct"]+=1
        if ov: self.m["overrides"]+=1
        if cr: self.m["critical"]+=1
        self.trust=min(1000,self.trust+(10 if ok else -50))
    def advance(self):
        m=self.m
        if m["total"]<1000: return False
        if m["correct"]/m["total"]>=0.98 and m["overrides"]/m["total"]<=0.02 and m["critical"]==0:
            stages=["audit","assist","automate"]; idx=stages.index(self.stage)
            if idx<2: self.stage=stages[idx+1]; return True
        return False
    def rollback(self):
        stages=["audit","assist","automate"]; idx=stages.index(self.stage)
        if idx>0: old=self.stage; self.stage=stages[idx-1]; self.trust=max(0,self.trust-200); return f"{old}->{self.stage}"
        return "already audit"

pa=PA()
for i in range(1005): pa.process(ok=(i%100!=50),ov=(i%200==0))
adv=pa.advance()
print(f"Progressive Autonomy: {pa.stage} trust={pa.trust} advanced={adv}")
print(f"  Accuracy: {pa.m['correct']/pa.m['total']:.1%} Overrides: {pa.m['overrides']/pa.m['total']:.1%}")
pa.stage="automate"; pa.trust=800
print(f"  Rollback: {pa.rollback()} trust={pa.trust}")
print(f"  Criteria: 98% accuracy, <2% override, 0 critical, 1000+ items")


</details>


---
## Exercise 8 (Challenge): Compliance Audit Trail
Design/architecture.


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
from datetime import datetime, timedelta

class Audit:
    def __init__(self,months=6): self.logs=[]; self.ret=timedelta(days=months*30)
    def log(self,action,model,conf,decision=None,reason=""):
        self.logs.append({"ts":datetime.now().isoformat()[:19],"action":action,"model":model,
            "confidence":conf,"decision":decision or "auto","reason":reason,
            "retain_until":(datetime.now()+self.ret).isoformat()[:10]})
    def search(self,**f):
        return [l for l in self.logs if all(l.get(k)==v for k,v in f.items())]

a=Audit(6)
for act,model,conf,dec,reason in [("search","sonnet-4.6",0.95,None,""),("send_email","sonnet-4.6",0.88,"approved","Looks good"),
    ("refund","sonnet-4.6",0.72,"approved","Within 7 days"),("delete","sonnet-4.6",0.65,"rejected","Need manager"),
    ("purchase","sonnet-4.6",0.80,"approved","Budget OK")]:
    a.log(act,model,conf,dec,reason)

print("Compliance Audit Trail:")
for l in a.logs: print(f"  [{l['confidence']:.2f}] {l['action']:<14} -> {l['decision']:>10} | {l['reason'][:20]}")
approved=a.search(decision="approved")
print(f"\n  Search(approved): {len(approved)} items")
print(f"  Retention: 6 months (until {a.logs[0]['retain_until']})")
print(f"  EU AI Act Art 26: 6+ months | FINRA: same as human | DPDP: prompts=personal data")


</details>
